# 04 — Weekly Picks

Run this every Tuesday/Wednesday to generate picks for the upcoming week.

**Workflow:** refresh data (01) → rebuild features (02) → run this notebook → bet only the edges past your threshold.

In [ ]:
import nflreadpy as nfl
import pandas as pd
import polars as pl
import pickle
from pathlib import Path

MODEL_DIR = Path('../models')
DATA_DIR  = Path('../data/processed')

with open(MODEL_DIR / 'margin_model.pkl', 'rb') as f:
    bundle = pickle.load(f)
model, feature_cols, elo_ratings = bundle['model'], bundle['features'], bundle['elo_ratings']

current_season = nfl.get_current_season()
current_week   = nfl.get_current_week()
print(f'Picking for Season {current_season} Week {current_week}')

## Get this week's games

In [ ]:
sched = nfl.load_schedules(seasons=[current_season]).to_pandas()
upcoming = sched[(sched['week'] == current_week) & (sched['game_type'] == 'REG')].copy()
print(f'{len(upcoming)} games this week')
upcoming[['game_id','gameday','home_team','away_team','spread_line','total_line','home_moneyline','away_moneyline']]

## Build features for upcoming games

In [ ]:
# Pull most recent team-week stats (already lagged in notebook 02)
tw = pd.read_parquet(DATA_DIR / 'team_week.parquet')
# Get each team's most recent stat row (latest season/week with data)
latest = tw.sort_values(['team','season','week']).groupby('team').tail(1)
stat_cols = [c for c in tw.columns if c not in ['season','week','team']]

home = latest.rename(columns={'team':'home_team', **{c: f'home_{c}' for c in stat_cols}})
away = latest.rename(columns={'team':'away_team', **{c: f'away_{c}' for c in stat_cols}})

g = upcoming.merge(home.drop(columns=['season','week']), on='home_team', how='left')
g = g.merge(away.drop(columns=['season','week']), on='away_team', how='left')

raw_stat_cols = ['off_epa_play','off_success_rate','off_explosive_rate',
                 'def_epa_play','def_success_rate','def_explosive_rate']
for c in raw_stat_cols:
    for w in ['l4','l8','ytd']:
        g[f'diff_{c}_{w}'] = g[f'home_{c}_{w}'] - g[f'away_{c}_{w}']

HFA = 55
g['home_elo'] = g['home_team'].map(elo_ratings).fillna(1500)
g['away_elo'] = g['away_team'].map(elo_ratings).fillna(1500)
g['elo_diff'] = g['home_elo'] - g['away_elo'] + HFA
g['rest_diff'] = g['home_rest'] - g['away_rest']

## Predict + rank edges

In [ ]:
X = g[feature_cols].copy()
g['pred_margin'] = model.predict(X)

# Convert margin → win prob (NFL margin std dev ≈ 13.5)
from scipy.stats import norm
g['home_win_prob'] = norm.cdf(g['pred_margin'] / 13.5)

# Edge vs. Vegas: spread_line > 0 means home favored. Model picks home if pred_margin > spread_line.
g['ats_edge'] = g['pred_margin'] - g['spread_line']
g['ats_pick'] = g['ats_edge'].apply(lambda x: 'HOME' if x > 0 else 'AWAY')
g['ats_pick_team'] = g.apply(lambda r: r['home_team'] if r['ats_pick']=='HOME' else r['away_team'], axis=1)
g['ats_edge_pts'] = g['ats_edge'].abs()

# Confidence tiers
def tier(e):
    if e >= 5: return 'STRONG'
    if e >= 3: return 'LEAN'
    if e >= 1.5: return 'WEAK'
    return 'PASS'
g['confidence'] = g['ats_edge_pts'].apply(tier)

picks = g[['gameday','home_team','away_team','spread_line','pred_margin',
           'home_win_prob','ats_pick_team','ats_edge_pts','confidence']].sort_values('ats_edge_pts', ascending=False)
picks

## How to use

- **STRONG / LEAN**: model disagrees with Vegas by 3+ points — your bettable spots.
- **WEAK**: small edge; ignore unless you also like other angles (injury news, weather).
- **PASS**: model agrees with Vegas — no edge.
- **Straight-up picks (survivor pools)**: sort by `home_win_prob` — pick teams >0.70 you haven't used.

Track every pick. After 200+ bets, compare your hit rate to 52.4% (break-even at -110 vig).